# Committor from MD on a 2D Toy Potential (Brownian or Langevin)

Runs long MD on a 2D toytis potential with exactly the toytis integrators, and fits the committor with optimalrcs. Default potential: `curved_channel:CurvedChannelDoubleWell`, a smooth double well whose channel bends and runs diagonally, so the committor depends on both coordinates and the isocommittor lines are curved and tilted relative to the order parameter y. It has no walls, kinks or flat intermediate region.

**Dynamics** (`DYNAMICS`):
- `brownian`: overdamped, x ← x + dt/(mγ) F + N(0, √(2 kT dt/(mγ))) per coordinate. The committor depends on positions only and is computed exactly (finite volumes, `toy2d.exact_committor_brownian`). Fit: NPq (`Committor`) on (y, x).
- `langevin`: underdamped, the BBK scheme of the toytis `ndLangevinEngine`. The committor depends on positions and velocities, q(y, x, v_y, v_x). Fit: NPNEq (`CommittorNE`) on (y, x, v_y, v_x), with a lag-Δ update if γ·Δ·dt_saved is small (notes §10.1). NPq is not valid here: the dynamics is reversible only with v → −v.

**References** (no fit involved):
- *MD outcome*: for every frame, the state the trajectory reaches next (1 = B, 0 = A). Its mean over a bin is the committor averaged over the frames in that bin, for either dynamics.
- *Exact* (Brownian only): the PDE solution, with A/B shifted by the mean overshoot for absorption checked every `STRIDE` steps.
- *Shooting*: the committor of a random sample of frames, from `N_SHOTS` new trajectories each (both dynamics; for Langevin the frame's velocity is kept).

Every figure is shown and saved to `FIG_DIR` (named after the potential and the MD settings); the fit's plots go to `FIG_DIR/fit_<estimator>/seed<s>/it<N>_<plot>.png`. The MD itself is cached in `FIG_DIR/md.npz`.

## 1. Imports

In [ ]:
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import cupy as cp

plt.rc('font', size=10)
plt.rc('axes', titlesize=12)
plt.rc('axes', labelsize=12)
plt.rc('legend', fontsize=9)

REPO = '/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/OP_improvement/optimalrcs'
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs
import seed_runs
import toy2d

## 2. Settings

| Name | Default | Meaning |
|---|---|---|
| `POTENTIAL`, `POT_KWARGS` | `curved_channel:CurvedChannelDoubleWell`, `{}` | toytis potential (`module:Class`) and its constructor arguments |
| `DYNAMICS` | `brownian` | `brownian` or `langevin` |
| `KT`, `DT`, `MASS`, `GAMMA` | 0.05, 0.005, 1, 10 (Brownian) / 2 (Langevin) | temperature, time step, mass, friction |
| `LAMBDA_A`, `LAMBDA_B` | 0.15, 0.85 | A: y ≤ λ_A, B: y ≥ λ_B |
| `N_WALKERS`, `N_STEPS`, `STRIDE`, `BURN_IN` | 128, 1,000,000, 50, 20,000 | independent trajectories, steps per walker, steps between saved frames (A/B are checked on saved frames), steps discarded at the start. 2.56M saved frames: NPq evaluates its basis on all frames at once, so much more does not fit an 8 GB GPU |
| `MD_ON_GPU` | False | integrate on the GPU (CuPy); the CPU is faster for ~100 walkers |
| `CVS` | (y, x) / (y, x, v_y, v_x) | fit CVs |
| `ESTIMATOR`, `FIT_LAG` | NPq, 1 / NPNEq, Δ | estimator; fit lag in saved frames (stopped windows, NPNEq only) |
| `FIT_SEEDS`, `WARMUP_ITER`, `WARMUP_ENVELOPE` | [0], 1000, 1 | seeds, and a warm-up with a wide envelope |
| `MAX_ITER`, `CHUNK`, `ENVELOPE`, `FIT_GAMMA`, `NY` | 6000, 2000, 0.01, 0.1, 6 | main fit; plots after every chunk |
| `N_SHOOT_POINTS`, `N_SHOTS` | 300, 200 | shooting reference |

With the defaults the potential has a 4 kT barrier; the Brownian MD makes about one A→B transition per 1.3·10⁵ steps (≈ 500 in 128 × 10⁶ steps) and takes about a minute.

In [ ]:
POTENTIAL = 'curved_channel:CurvedChannelDoubleWell'
POT_KWARGS = {}                     # e.g. dict(Vb=0.25, b=0.3, a=1.0)
DYNAMICS = 'brownian'               # 'brownian' or 'langevin'
KT, DT, MASS = 0.05, 0.005, 1.0
GAMMA = 10.0 if DYNAMICS == 'brownian' else 2.0
LAMBDA_A, LAMBDA_B = 0.15, 0.85
Y_RANGE, X_RANGE = None, None      # plot/grid ranges (None: from the potential's grid())

N_WALKERS, N_STEPS, STRIDE, BURN_IN, MD_SEED = 128, 1_000_000, 50, 20_000, 0   # 2.56M saved frames
MD_ON_GPU = False

BROWNIAN = DYNAMICS == 'brownian'
CVS = ('y', 'x') if BROWNIAN else ('y', 'x', 'vy', 'vx')
ESTIMATOR = 'NPq' if BROWNIAN else 'NPNEq'
# lag in saved frames: gamma * lag * (DT * STRIDE) of about 0.05 (lag 1 if the saved frames are already that far apart)
FIT_LAG = 1 if BROWNIAN else max(1, int(round(0.05 / (GAMMA * DT * STRIDE))))
FIT_SEEDS = [0]
WARMUP_ITER, WARMUP_ENVELOPE = 1000, 1.0
MAX_ITER, CHUNK = 6000, 2000
ENVELOPE, FIT_GAMMA, NY = 0.01, 0.1, 6
N_SHOOT_POINTS, N_SHOTS = 300, 200
MIN_COUNT = 20                      # frames a bin needs to be shown

pot = toy2d.load_potential(POTENTIAL, POT_KWARGS)
pot_tag = POTENTIAL.split(':')[1] + ''.join(f'_{k}{v}' for k, v in POT_KWARGS.items())
md_tag = f'{DYNAMICS}_kT{KT:g}_g{GAMMA:g}_dt{DT:g}_s{STRIDE}_w{N_WALKERS}_n{N_STEPS}_seed{MD_SEED}'
FIG_DIR = f'{REPO}/examples/figures/nb_toy2d_md/{pot_tag}/{md_tag}'
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(name, fig=None):
    (fig or plt.gcf()).savefig(f'{FIG_DIR}/{name}.png', dpi=110, bbox_inches='tight')

print(repr(pot))
print('figures and cache:', FIG_DIR)
# === SETTINGS END ===

## 3. The Potential

V/kT on a grid with contours, the states A and B, and (if the potential provides them) the channel centre line and the stationary points. Right: V/kT along the centre line.

In [ ]:
if Y_RANGE is None and hasattr(pot, 'grid'):
    _g = pot.grid()
    Y_RANGE, X_RANGE = (_g[0][0], _g[0][-1]), (_g[1][0], _g[1][-1])
gy = np.linspace(*Y_RANGE, 241)
gx = np.linspace(*X_RANGE, 161)
V = toy2d.potential_grid(pot, gy, gx)
V_kT = (V - V.min()) / KT
LEVELS = np.arange(0, 12.5, 1.0)      # potential contours in kT, reused on all maps
CONTOURS = (gy, gx, V_kT, LEVELS)

def draw_potential(ax, lines=True):
    ax.contour(gy, gx, V_kT.T, levels=LEVELS, colors='0.55', linewidths=0.5)
    if lines:
        for lam in (LAMBDA_A, LAMBDA_B):
            ax.axvline(lam, color='k', lw=0.8)
    ax.set(xlim=Y_RANGE, ylim=X_RANGE, xlabel='$y$ (order parameter)', ylabel='$x$')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={'width_ratios': [1.6, 1]})
im = ax1.contourf(gy, gx, np.clip(V_kT, None, 12).T, 40, cmap='viridis')
draw_potential(ax1)
fig.colorbar(im, ax=ax1, label='$V/kT$ (clipped at 12)')
if hasattr(pot, 'centre_line'):
    ax1.plot(gy, pot.centre_line(gy), 'w--', lw=1, label='channel centre')
if hasattr(pot, 'stationary_points'):
    sp = pot.stationary_points()
    for name, (sy, sx, sv) in sp.items():
        ax1.plot(sy, sx, 'r*' if name == 'saddle' else 'wo', ms=10)
        ax1.annotate(f'{name} ({(sv - V.min()) / KT:.1f} kT)', (sy, sx), color='w', xytext=(5, 5), textcoords='offset points')
    print(pot.barrier_info())
ax1.set_title(f'{pot_tag}: $V/kT$, A: $y \\leq {LAMBDA_A}$, B: $y \\geq {LAMBDA_B}$')
if hasattr(pot, 'centre_line'):
    cy = np.linspace(*Y_RANGE, 400)
    vc = (pot.potential_and_force_xy(cy, pot.centre_line(cy))[0] - V.min()) / KT
    ax2.plot(cy, vc)
    ax2.set(xlabel='$y$', ylabel='$V/kT$ on the centre line', ylim=(0, min(15, vc.max())))
else:
    ax2.plot(gy, V_kT.min(axis=1))
    ax2.set(xlabel='$y$', ylabel='min$_x$ $V/kT$')
for lam in (LAMBDA_A, LAMBDA_B):
    ax2.axvline(lam, color='k', lw=0.8)
ax2.grid(alpha=0.3)
fig.tight_layout()
save_fig('potential')
plt.show()

## 4. Run the MD

`N_WALKERS` independent trajectories (half started in each well), integrated together. After `BURN_IN` steps every `STRIDE`-th step is saved (positions and velocities; for Brownian dynamics the "velocity" is the random kick of the step, as toytis stores it). Cached in `FIG_DIR/md.npz`.

In [ ]:
cache = f'{FIG_DIR}/md.npz'
if os.path.exists(cache):
    md = dict(np.load(cache))
    print('loaded', cache)
else:
    t0 = time.time()
    md = toy2d.run_md(pot, N_WALKERS, N_STEPS, STRIDE, DT, KT, GAMMA, MASS, high_friction=BROWNIAN,
                      burn_in=BURN_IN, seed=MD_SEED, xp=cp if MD_ON_GPU else np)
    np.savez(cache, **md)
    print(f'MD done in {time.time() - t0:.0f} s')
y, x, vy, vx = md['y'], md['x'], md['vy'], md['vx']       # (walker, saved frame)
n_w, n_f = y.shape
print(f'{n_w} walkers x {n_f} saved frames = {n_w * n_f:,} frames ({N_STEPS * DT:g} time units per walker)')

n_ab, n_ba = toy2d.count_transitions(y, LAMBDA_A, LAMBDA_B)
# fraction of the time last in A (for the rate per time spent on the A side)
last = np.where(y <= LAMBDA_A, 0, np.where(y >= LAMBDA_B, 1, -1)).astype(float)
last[last < 0] = np.nan
for w in range(n_w):                      # forward fill: the state visited last
    s = last[w]; idx = np.where(~np.isnan(s), np.arange(n_f), 0)
    np.maximum.accumulate(idx, out=idx); last[w] = s[idx]
h_a = np.nanmean(last == 0)
steps_per_ab = N_WALKERS * N_STEPS * h_a / max(n_ab.sum(), 1)
print(f'A->B transitions {n_ab.sum()}, B->A {n_ba.sum()}; time last in A {h_a:.2f}; '
      f'MD: {steps_per_ab:.3g} steps on the A side per A->B transition')

ref = None
if BROWNIAN:
    ref = toy2d.exact_committor_brownian(pot, KT, GAMMA, DT, LAMBDA_A, LAMBDA_B, stride=STRIDE, mass=MASS,
                                         y_range=Y_RANGE, x_range=X_RANGE)
    print(f'TPT (exact, continuous time): {ref["steps_per_ab"]:.3g} steps per A->B; '
          f'MD / TPT = {steps_per_ab / ref["steps_per_ab"]:.2f} (+- {1 / np.sqrt(max(n_ab.sum(), 1)):.2f})')

## 5. MD Checks

- Traces y(t) of a few walkers with the state boundaries.
- Sampled free energy F(y, x) = −kT ln p against the potential: at equilibrium F − V is constant (up to noise where few frames are).
- Marginal F(y) against the Boltzmann marginal −kT ln ∫ e^{−V/kT} dx.
- Langevin only: the velocity distributions against Maxwell–Boltzmann.
- Transition paths (from the last frame in A to the first in B) on the potential, and their durations.

In [ ]:
t_axis = np.arange(n_f) * STRIDE * DT
fig, axes = plt.subplots(4, 1, figsize=(14, 7), sharex=True)
for ax, w in zip(axes, range(4)):
    ax.plot(t_axis, y[w], lw=0.4)
    for lam in (LAMBDA_A, LAMBDA_B):
        ax.axhline(lam, color='k', lw=0.8, ls='--')
    ax.set(ylabel=f'$y$, walker {w}')
axes[-1].set(xlabel='time')
fig.tight_layout()
save_fig('md_traces')
plt.show()

ey, ex = np.linspace(*Y_RANGE, 121), np.linspace(*X_RANGE, 81)
H = np.histogram2d(y.ravel(), x.ravel(), [ey, ex])[0]
my, mx = (ey[1:] + ey[:-1]) / 2, (ex[1:] + ex[:-1]) / 2
F = np.where(H > 0, -np.log(np.maximum(H, 1)), np.nan)
Vb = toy2d.potential_grid(pot, my, mx) / KT
D = F - Vb
D -= np.nanmedian(D[H >= 50])
fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
for ax, (M, title, cmap, lim) in zip(axes, [(F - np.nanmin(F), '$F/kT$ from the MD', 'viridis', (0, 12)),
                                           (Vb - Vb.min(), '$V/kT$', 'viridis', (0, 12)),
                                           (np.where(H >= 50, D, np.nan), '$(F - V)/kT$ (bins with $\\geq$ 50 frames)', 'PuOr', (-0.5, 0.5))]):
    im = ax.pcolormesh(ey, ex, M.T, cmap=cmap, vmin=lim[0], vmax=lim[1])
    draw_potential(ax)
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
fig.tight_layout()
save_fig('md_free_energy')
plt.show()

hy = np.histogram(y.ravel(), ey)[0]
Fy = -np.log(np.where(hy > 0, hy, np.nan))
Vfine = toy2d.potential_grid(pot, my, np.linspace(*X_RANGE, 400))
Fy_b = -np.log(np.exp(-(Vfine - Vfine.min()) / KT).sum(axis=1))
off = np.nanmin(Fy) - np.nanmin(Fy_b)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(my, Fy - np.nanmin(Fy), label='MD')
ax.plot(my, Fy_b + off - np.nanmin(Fy), 'k--', label='Boltzmann marginal')
for lam in (LAMBDA_A, LAMBDA_B):
    ax.axvline(lam, color='k', lw=0.8)
ax.set(xlabel='$y$', ylabel='$F(y)/kT$', ylim=(0, 10))
ax.legend()
ax.grid(alpha=0.3)
save_fig('md_free_energy_y')
plt.show()

if not BROWNIAN:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    vgrid = np.linspace(-5, 5, 200) * np.sqrt(KT / MASS)
    for ax, (v, name) in zip(axes, [(vy, '$v_y$'), (vx, '$v_x$')]):
        ax.hist(v.ravel()[::7], bins=100, density=True, alpha=0.6, label=f'MD, $\\langle v^2 \\rangle$ = {np.mean(v ** 2):.4f}')
        ax.plot(vgrid, np.exp(-MASS * vgrid ** 2 / (2 * KT)) / np.sqrt(2 * np.pi * KT / MASS), 'k--', label=f'Maxwell, kT/m = {KT / MASS:g}')
        ax.set(xlabel=name)
        ax.legend()
    fig.tight_layout()
    save_fig('md_velocities')
    plt.show()

# transition paths: from the last A frame to the next B frame
paths, durations = [], []
for w in range(n_w):
    st = np.where(y[w] <= LAMBDA_A, 0, np.where(y[w] >= LAMBDA_B, 1, -1))
    idx = np.flatnonzero(st >= 0)
    for a, b in zip(idx[:-1], idx[1:]):
        if st[a] == 0 and st[b] == 1:
            durations.append((b - a) * STRIDE * DT)
            if len(paths) < 40:
                paths.append((y[w, a:b + 1], x[w, a:b + 1]))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={'width_ratios': [1.6, 1]})
ax1.contourf(gy, gx, np.clip(V_kT, None, 12).T, 40, cmap='Greys_r', alpha=0.6)
draw_potential(ax1)
for py, px in paths:
    ax1.plot(py, px, lw=0.6, alpha=0.8)
ax1.set_title(f'{len(paths)} of {len(durations)} A$\\to$B transition paths')
if durations:
    ax2.hist(durations, bins=40)
    ax2.set(xlabel='transition path time', ylabel='paths', title=f'mean {np.mean(durations):.3g} ({np.mean(durations) / DT:.0f} steps)')
fig.tight_layout()
save_fig('md_transition_paths')
plt.show()

## 6. Reference Committors

The MD outcome per frame (which state comes next), the exact committor (Brownian), and their maps. For Langevin the committor also depends on the velocities: the (y, v_y) and (y, v_x) maps show that dependence in the MD outcome.

The shooting reference runs `N_SHOTS` trajectories from each of `N_SHOOT_POINTS` random frames between A and B (on the GPU). For Brownian dynamics it also checks the PDE solution.

In [ ]:
Y, X, VY, VX = (a.ravel().astype(np.float64) for a in (y, x, vy, vx))
i_traj = np.repeat(np.arange(n_w), n_f)
t_traj = np.tile(np.arange(n_f) * float(STRIDE), n_w)
A, B = Y <= LAMBDA_A, Y >= LAMBDA_B
free = ~(A | B)
outcome = toy2d.next_outcome(y, LAMBDA_A, LAMBDA_B).ravel().astype(np.float64)
known = ~np.isnan(outcome)
q_exact = toy2d.interp_grid(ref, Y, X) if ref is not None else None
q_ref = q_exact if q_exact is not None else np.where(known, outcome, 0.0)   # reference for the fit plots
ref_name = 'exact' if q_exact is not None else 'MD outcome'
ref_mask = free & known                                                       # frames compared with the reference
cv = {'y': Y, 'x': X, 'vy': VY, 'vx': VX}

e_y = np.linspace(LAMBDA_A, LAMBDA_B, 71)
e_x = np.linspace(*np.percentile(X[free], [0.5, 99.5]), 41)
vmax = 3.5 * np.sqrt(KT / MASS)
e_v = np.linspace(-vmax, vmax, 41)
planes = [seed_runs.plane(Y, e_y, '$y$', X, e_x, '$x$', CONTOURS)]
if not BROWNIAN:
    planes += [seed_runs.plane(Y, e_y, '$y$', VY, e_v, '$v_y$'), seed_runs.plane(Y, e_y, '$y$', VX, e_v, '$v_x$')]

def show_maps(maps, title, name):
    """maps: list of (plane, values per frame, label, cmap, limits)."""
    fig, axes = plt.subplots(1, len(maps), figsize=(6.5 * len(maps), 4.5), squeeze=False)
    for ax, (p, val, label, cmap, lim) in zip(axes[0], maps):
        bm = seed_runs.BinnedMean(p['c'], p['edges'], ref_mask, MIN_COUNT)
        M = bm(val)
        (e1, e2), (l1, l2) = p['edges'], p['labels']
        im = ax.pcolormesh(e1, e2, M.T, cmap=cmap, vmin=lim[0], vmax=lim[1])
        if p['contours'] is not None:
            g1, g2, VV, lev = p['contours']
            ax.contour(g1, g2, VV.T, levels=lev, colors='0.5', linewidths=0.5)
        if lim[0] == 0:
            m1, m2 = (e1[1:] + e1[:-1]) / 2, (e2[1:] + e2[:-1]) / 2
            ax.contour(m1, m2, np.nan_to_num(M, nan=0.5).T, levels=[0.1, 0.5, 0.9], colors='k', linewidths=1, linestyles='--')
        ax.set(xlabel=l1, ylabel=l2, title=label, xlim=(e1[0], e1[-1]), ylim=(e2[0], e2[-1]))
        fig.colorbar(im, ax=ax)
    fig.suptitle(title)
    fig.tight_layout()
    save_fig(name, fig)
    plt.show()

maps = [(p, outcome, f'MD outcome ({p["labels"][0]}, {p["labels"][1]})', 'coolwarm', (0, 1)) for p in planes]
if q_exact is not None:
    maps += [(planes[0], q_exact, 'exact committor', 'coolwarm', (0, 1)),
             (planes[0], outcome - q_exact, 'MD outcome $-$ exact', 'PuOr', (-0.15, 0.15))]
show_maps(maps, 'Model-free committor of the MD frames' + (' and the exact committor' if q_exact is not None else ''),
          'reference_maps')

# shooting reference at random frames between A and B
rng = np.random.default_rng(1)
shoot_idx = rng.choice(np.flatnonzero(free), size=min(N_SHOOT_POINTS, free.sum()), replace=False)
t0 = time.time()
q_shoot, se_shoot = toy2d.shoot(pot, Y[shoot_idx], X[shoot_idx], VY[shoot_idx], VX[shoot_idx], N_SHOTS, DT, KT, GAMMA,
                                LAMBDA_A, LAMBDA_B, MASS, BROWNIAN, xp=cp, check_every=STRIDE, progress=False)
print(f'shooting: {len(shoot_idx)} frames x {N_SHOTS} trajectories in {time.time() - t0:.0f} s, '
      f'mean standard error {np.nanmean(se_shoot):.3f}')
if q_exact is not None:
    d = q_exact[shoot_idx] - q_shoot
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.errorbar(q_shoot, q_exact[shoot_idx], xerr=se_shoot, fmt='.', ms=3, alpha=0.6)
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set(xlabel='shooting', ylabel='exact (PDE)',
           title=f'PDE vs shooting: RMS {np.sqrt(np.mean(d ** 2)):.3f}, mean {d.mean():+.4f}\n(shooting error {np.nanmean(se_shoot):.3f})')
    save_fig('exact_vs_shooting')
    plt.show()

## 7. Fit the Committor

NPq (`Committor`) for Brownian dynamics on (y, x); NPNEq (`CommittorNE`) for Langevin on (y, x, v_y, v_x), with the stopped-window lag-`FIT_LAG` update when `FIT_LAG` > 1 (`toy2d.lag_update`). Each walker is one trajectory (`i_traj`), with A and B as traps for the Z_q metrics (`set_boundary_traps`), so Z_q converges to N_AB.

After the warm-up and every `CHUNK` iterations, `seed_runs.run_seeds` shows and saves the library plots (`metrics`, `feps`, `obs_pred`, `obs_pred_log`) and the fitted q against the CVs next to the reference (exact for Brownian, MD outcome for Langevin), with the RMS of fit − reference per bin.

In [ ]:
X_cv = cp.asarray(np.column_stack([cv[c] for c in CVS]))
mid_y = (e_y[1:] + e_y[:-1]) / 2
prof_ref = seed_runs.BinnedMean((Y,), (e_y,), ref_mask, MIN_COUNT)(q_ref)   # mean reference q per y bin
fit_dir = f'{FIG_DIR}/fit_{ESTIMATOR}' + (f'_lag{FIT_LAG}' if FIT_LAG > 1 else '')

def make_q():
    cls = optimalrcs.Committor if ESTIMATOR == 'NPq' else optimalrcs.CommittorNE
    q = cls(boundary0=A, boundary1=B, i_traj=i_traj, t_traj=t_traj)
    q.set_boundary_traps()
    return q

if ESTIMATOR == 'NPq':
    extra_kw = {}
else:
    extra_kw = toy2d.lag_update(optimalrcs, i_traj, A | B, None, FIT_LAG)
METRICS = ['delta_r2', 'max_sd_zq', 'auc', 'delta_x']
PRINT_STEP = min(500, CHUNK, WARMUP_ITER or CHUNK)   # metrics are computed every PRINT_STEP iterations
fit_kw = dict(ny=NY, print_step=PRINT_STEP, metrics_print=('iter', 'delta_r2', 'max_sd_zq', 'auc', 'delta_x', 'time_elapsed'),
              **extra_kw)

def report(q):
    r = cp.asnumpy(q.r_traj)
    trans = ref_mask & (q_ref > 0.05) & (q_ref < 0.95)
    bm = seed_runs.BinnedMean(planes[0]['c'], planes[0]['edges'], ref_mask, MIN_COUNT)
    Qf, Qr = bm(r), bm(q_ref)
    tb = (Qr > 0.05) & (Qr < 0.95)
    out = f'| RMS(fit - {ref_name}) per bin {np.sqrt(np.nanmean((Qf - Qr)[tb] ** 2)):.4f}'
    if q_exact is None:      # the MD outcome per bin is a mean of 0/1 values: its own noise (frames are correlated, so at least this)
        out += f' (noise of the reference >= {np.sqrt(np.nanmean((Qr * (1 - Qr) / np.maximum(bm.count, 1))[tb])):.4f})'
    if q_exact is not None:
        out += f', per frame {np.sqrt(np.mean((r - q_exact)[trans] ** 2)):.4f}'
    return out + f' | vs shooting {np.sqrt(np.nanmean((r[shoot_idx] - q_shoot) ** 2)):.4f}'

results, q = seed_runs.run_seeds(
    optimalrcs, make_q,
    comp_y=lambda: X_cv[:, np.random.randint(X_cv.shape[1])],
    fit_kw=fit_kw, seeds=FIT_SEEDS, max_iter=MAX_ITER, chunk=CHUNK, out_dir=fit_dir,
    envelope=ENVELOPE, gamma=FIT_GAMMA,
    warmup_iter=WARMUP_ITER, warmup_envelope=WARMUP_ENVELOPE, warmup_gamma=FIT_GAMMA,
    plots=('metrics', 'feps', 'obs_pred', 'obs_pred_log'), plot_options=dict(metrics=METRICS),
    extra_plot={'q_map': seed_runs.committor_map(planes[:1], mask=ref_mask, q_ref=q_ref, min_count=MIN_COUNT),
                'q_profile': seed_runs.committor_profile(Y, e_y, '$y$', mask=ref_mask, ref=(mid_y, prof_ref))},
    report=report,
)
r = cp.asnumpy(q.r_traj)

## 8. Results

- Fit, reference and fit − reference on every plane (for Langevin also (y, v_y) and (y, v_x) against the MD outcome).
- Isocommittor lines of the fit (solid) and the reference (dashed) on the potential.
- Calibration against the MD outcomes: per bin of fitted q, the fraction of frames that reach B next (model-free, valid for both dynamics).
- Fit against the shooting reference at the shot frames.
- Seeds: the fits of all `FIT_SEEDS` against each other (if more than one).

In [ ]:
maps = []
for p in planes:
    maps.append((p, r, f'fit ({p["labels"][0]}, {p["labels"][1]})', 'coolwarm', (0, 1)))
    ref_p = q_exact if (q_exact is not None and p is planes[0]) else outcome
    name_p = 'exact' if (q_exact is not None and p is planes[0]) else 'MD outcome'
    maps.append((p, ref_p, name_p, 'coolwarm', (0, 1)))
    maps.append((p, r - ref_p, f'fit $-$ {name_p}', 'PuOr', (-0.15, 0.15)))
for k in range(0, len(maps), 3):
    show_maps(maps[k:k + 3], f'Fit ({ESTIMATOR}) against the reference', f'result_maps_{k // 3}')

# isocommittor lines on the potential: the fit as the mean q per bin (bins with >= 5 frames),
# the exact committor (Brownian) on its own grid, else the MD outcome per bin
p = planes[0]
bm5 = seed_runs.BinnedMean(p['c'], p['edges'], ref_mask, 5)
(e1, e2) = p['edges']
m1, m2 = (e1[1:] + e1[:-1]) / 2, (e2[1:] + e2[:-1]) / 2
fig, ax = plt.subplots(figsize=(10, 5))
draw_potential(ax)
levels = [0.1, 0.25, 0.5, 0.75, 0.9]
cs1 = ax.contour(m1, m2, bm5(r).T, levels=levels, colors='C3', linewidths=1.8)
if ref is not None:
    ax.contour(ref['y'], ref['x'], ref['q'].T, levels=levels, colors='C0', linewidths=1.2, linestyles='--')
else:
    ax.contour(m1, m2, bm5(q_ref).T, levels=levels, colors='C0', linewidths=1.2, linestyles='--')
ax.clabel(cs1, fmt='%.2f', fontsize=8)
ax.plot([], [], 'C3', label='fit (mean per bin)')
ax.plot([], [], 'C0--', label=ref_name)
ax.legend()
ax.set_title('isocommittor lines 0.1, 0.25, 0.5, 0.75, 0.9 on the potential (grey: $V/kT$ = 0, 1, 2, ...)')
save_fig('isocommittor')
plt.show()

# calibration against the MD outcome
eb = np.linspace(0, 1, 26)
k = np.clip(np.digitize(r[ref_mask], eb) - 1, 0, 24)
n_b = np.bincount(k, minlength=25)
obs = np.bincount(k, weights=outcome[ref_mask], minlength=25) / np.maximum(n_b, 1)
mid = (eb[1:] + eb[:-1]) / 2
# frames of one walker are correlated: the error bar uses the number of transitions-sized blocks, roughly
err = np.sqrt(obs * (1 - obs) / np.maximum(n_b / 100, 1))
fig, ax = plt.subplots(figsize=(5.5, 5))
ok = n_b > 100
ax.errorbar(mid[ok], obs[ok], yerr=err[ok], fmt='o-', ms=4)
ax.plot([0, 1], [0, 1], 'k--')
ax.set(xlabel='fitted $q$', ylabel='fraction reaching B next (MD)', title='calibration against the MD outcomes')
save_fig('calibration')
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.errorbar(q_shoot, r[shoot_idx], xerr=se_shoot, fmt='.', ms=3, alpha=0.6)
ax.plot([0, 1], [0, 1], 'k--')
d = r[shoot_idx] - q_shoot
ax.set(xlabel='shooting', ylabel='fit', title=f'fit vs shooting at {len(shoot_idx)} frames\nRMS {np.sqrt(np.nanmean(d ** 2)):.3f} (shooting error {np.nanmean(se_shoot):.3f})')
save_fig('fit_vs_shooting')
plt.show()

if len(results) > 1:
    seeds = list(results)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    base = results[seeds[0]]['r']
    for s in seeds[1:]:
        ax.plot(base[free][::200], results[s]['r'][free][::200], '.', ms=2, label=f'seed {s}')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set(xlabel=f'seed {seeds[0]}', ylabel='other seeds', title='fits with different seeds (every 200th frame)')
    ax.legend()
    save_fig('seeds')
    plt.show()

## 9. Save

The frames, the fit, the MD outcome, the exact committor (Brownian) and the shooting reference, as `FIG_DIR/committor.npz`.

In [ ]:
np.savez(f'{FIG_DIR}/committor.npz', y=Y.astype(np.float32), x=X.astype(np.float32), vy=VY.astype(np.float32),
         vx=VX.astype(np.float32), i_traj=i_traj, q_fit=r.astype(np.float32), outcome=outcome.astype(np.float32),
         q_exact=(q_exact if q_exact is not None else np.full(len(Y), np.nan)).astype(np.float32),
         shoot_idx=shoot_idx, q_shoot=q_shoot, se_shoot=se_shoot, lambda_a=LAMBDA_A, lambda_b=LAMBDA_B,
         settings=repr(dict(potential=repr(pot), dynamics=DYNAMICS, kT=KT, dt=DT, gamma=GAMMA, stride=STRIDE,
                            estimator=ESTIMATOR, fit_lag=FIT_LAG, cvs=CVS, envelope=ENVELOPE, fit_gamma=FIT_GAMMA)))
print('saved', f'{FIG_DIR}/committor.npz')